# Leave-one-recording-out — does the model generalise to an unseen recording?

The third analysis, alongside `Windowed_Analysis.ipynb` (*when* is the information there) and
`Full_Epoch_Analysis.ipynb` (*how well* does a whole trial classify). Both of those split over
**trials**, blind to which recording each came from — so every recording is represented in the
training set. **This one holds out a whole recording.**

Every step lives in `src/session_batch.py`; shared machinery is in `src/analysis_common.py`.
**All settings live in the Parameters cell below** — nothing under it needs editing.

| stage | call | writes |
|---|---|---|
| 1 | `run_session_batch()` | `Analysis/<label>/Session/Metrics/Individuals/`, `.../Figures/<subject>/` |
| 2 | `summarize_session()` | `.../Session/Metrics/Group/`, `.../session_summary.{csv,json}` |

## Why this exists

**1. Cross-session generalisation.** The live system runs on a session it has never seen.
A trial-level split measures nothing about that.

**2. The centering question, settled structurally.** `EEG_Preprocessing` centres *per file*.
Under a trial-level split, that file's trials are spread across folds, so the class mean
subtracted from a test trial was computed *including that trial* — the pooled-centering leak.
Hold out a whole recording and the centered-test condition isn't merely leaky, it is
**impossible**: you cannot subtract a class mean from an unseen recording without knowing its
labels.

So under `cross`, mode `c2u` is **leak-free by construction** — the training recordings' class
means are estimated inside files containing no test trials at all.

## The two splits

| split | how | folds |
|---|---|---|
| `cross` | `LeaveOneGroupOut` by recording — train on all but one, test on it | n_recordings (3–5) |
| `within` | `StratifiedKFold` over trials, recording-blind, **same fold count** | n_recordings |

`within` is a *control*, not a result. Leave-one-recording-out trains on ⅔–⅘ of the data, so a
drop would partly just be less training data. Matching the fold count matches the training
fraction, which makes the difference interpretable:

> **`generalisation_gap = within − cross`** — the cost of an unseen recording, training size held
> constant. This is the number the module exists to produce.

## The three modes

| mode | train | test | meaning |
|---|---|---|---|
| **`c2u`** | per-file centered | raw | **reportable — leak-free by construction under `cross`** |
| `u2u` | raw | raw | no-centering reference |
| `c2c` | per-file centered | centered | sizes the leak; **not achievable live** |

## Read before quoting anything

- The held-out unit is one **XDF recording**, not necessarily one session — several may be
  same-day runs. These are a **lower bound** on true cross-session difficulty.
- 3–5 folds makes per-subject estimates noisy. Lead with the between-subject spread.
- The held-out recording can have a different class balance, so macro F1 sits beside accuracy.

All of this is written into `session_summary.json['caveats']`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import warnings; warnings.filterwarnings('ignore')

from src.session_batch import *

# src.preprocessing forces %matplotlib qt at import time, so set the backend after it.
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')

print('subjects          :', list_subjects())
print('electrode groups  :', list(ELECTRODE_GROUPS))
print('splits            :', list(SPLITS))
print('modes             :', list(MODES), '| definitions:', list(DEFINITIONS))
print('window            :', FULL_EPOCH_WINDOW)

## Parameters — the only cell you need to edit

Everything the pipeline reads, spelled out. `default_params()` seeds it, then each key is set
explicitly so the value is visible and editable here rather than in `src/`.

`check_params` rejects a misspelled key (which would otherwise be silently ignored) and the
combinations that fail deep inside preprocessing; `describe_params` marks with `*` whatever
differs from the library default.

**`LABEL` matters.** The output tree is named from `desired_events` alone, and group results are
merged by subject rather than replaced — so re-running with a different filter band or electrode
set would mix the two runs. `check_params` warns when the tree on disk was built with different
params; set `LABEL` to a new name when it does.

**Definition `single` overrides three keys**, exactly as in `Full_Epoch_Analysis.ipynb`: it
rewrites `classifier_window_s`, `classifier_window_e` and `windowed_prediction_params` to span
`FULL_EPOCH_WINDOW`. Those three are `vote`-only knobs.

In [ ]:
# ══ PARAMETERS ═══════════════════════════════════════════════════════════════════
ELECTRODE_GROUP_NAMES = 'FC+C+CP+P'   # groups available: printed by the cell above
LABEL = None                          # Analysis/<label>/ tree; None -> named from
                                      # desired_events. Set it when you change any
                                      # other param (see the note above).

PARAMS = default_params(electrode_group_names=ELECTRODE_GROUP_NAMES)

# ── active ───────────────────────────────────────────────────────────────────────
PARAMS['desired_events'] = ['MiddleHand', 'LeftHand', 'RightHand', 'FixatedRest']
PARAMS['PerformAvgRef']  = True     # average re-reference
PARAMS['AddRefChannel']  = False    # reconstruct FCz (the online reference); needs
                                    # PerformAvgRef. False -> no FCz, 35 picks not 36
PARAMS['CenterByClass']  = True     # per-class mean removal (label-dependent, so the
                                    # live loop cannot apply it). HERE the centered /
                                    # uncentered epoch variants override it per build
PARAMS['PerformCsd']     = False    # current-source-density transform
PARAMS['PerformAsr']     = False    # Artifact Subspace Reconstruction on the continuous
                                    # data, BEFORE ICA and before the band-pass. Off =
                                    # every existing result unchanged. Flipping it
                                    # rebuilds the epoch cache; set LABEL too if you
                                    # want the ASR and non-ASR trees side by side
PARAMS['asr_cutoff']     = 20       # rejection threshold, SD of the clean calibration
                                    # data. Lower = more aggressive
PARAMS['asr_max_bad_chans'] = 0.1   # max bad-channel fraction a calibration window may have
PARAMS['asr_backend']    = 'asrpy'  # 'asrpy' (euclid only) or 'meegkit'. meegkit
                                    # must be 0.1.7: newer ones need pyriemann>=0.7,
                                    # and 0.12 needs numpy 2, which breaks mne 1.6.1
PARAMS['asr_method']     = 'euclid' # 'euclid', or 'riemann' (Blum et al. 2019) which
                                    # requires asr_backend='meegkit' - asrpy accepts
                                    # 'riemann' but silently runs euclid, so we reject it
PARAMS['asr_estimator']  = 'lwf'    # meegkit only. 'riemann' REQUIRES a regularising
                                    # estimator: the average reference makes the block
                                    # covariances singular and the riemannian mean needs
                                    # positive definite input ('scm' fails outright)
PARAMS['filter_method']  = 'iir'
PARAMS['LowPass']        = 8        # band-pass low edge, Hz
PARAMS['HighPass']       = 32       # band-pass high edge, Hz
PARAMS['epoch_tmin']     = -5       # epoch crop, s relative to cue
PARAMS['epoch_tmax']     = 6
PARAMS['classifier_window_s'] = 0.2                                    # 'vote' only
PARAMS['classifier_window_e'] = 4                                      # 'vote' only
PARAMS['windowed_prediction_params'] = {'win_len': 2, 'win_step': 0.25}  # 'vote' only
PARAMS['augmentation_params']        = {'win_len': 0, 'win_step': 0.25}  # 0 = off
PARAMS['pipeline_name'] = 'ts+FGDA'

# ── read only by the CSP / FBCSP pipelines - inert while pipeline_name is ts+FGDA ─
PARAMS['n_components']       = 8
PARAMS['n_components_fbcsp'] = 8
PARAMS['filters_bands']      = [[7, 12], [12, 20], [20, 28], [28, 35]]

# ── set by the pipeline itself; assigning them here has NO effect ────────────────
#   bad_electrodes              per subject, from get_subject_bad_electrodes
#   events_trigger_dict         per subject, from epochs.event_id
#   epoch_tmins_and_maxes_grid  vestigial - read nowhere

# ── what to run ──────────────────────────────────────────────────────────────────
SUBJECTS        = None    # None = all; or ['BA', 'AEH']
RUN_SPLITS      = None    # None = both; or ['cross'] to skip the control
RUN_MODES       = None    # None = all three; or ['c2u', 'u2u'] to skip leak-sizing
RUN_DEFINITIONS = None    # None = both; or ['single'] / ['vote']
FORCE           = False   # ignore the epoch cache and rebuild
SAVE_FIGS       = True

# ─────────────────────────────────────────────────────────────────────────────────
paths = project_paths(label=LABEL, params_dict=PARAMS)
check_params(PARAMS, label=LABEL, paths=paths)
describe_params(PARAMS)

print(f"\nanalysis root : {paths.analysis_root}")
print(f"session out   : {paths.out}")
print(f"shared cache  : {paths.cache}")
print()
session_status(paths=paths)

## Stage 1 — run every subject

For each subject: load both epoch variants from the shared cache, read the per-trial
`recording` provenance, balance once, then run every split × definition × mode cell off the
**same folds**.

The first run rebuilds any cache that predates recording provenance — file boundaries are not
recoverable from concatenated epochs, so the provenance has to come from the build step. That
rebuild is data-preserving; only metadata is added.

A failing subject is reported and skipped; it never aborts the batch. Rerun the failures by
setting `SUBJECTS` in the Parameters cell.

In [ ]:
out = run_session_batch(SUBJECTS, RUN_SPLITS, RUN_MODES, RUN_DEFINITIONS, force=FORCE,
                        save_figs=SAVE_FIGS, params_dict=PARAMS, paths=paths,
                        label=LABEL)

## Stage 2 — group statistics and the summary

Already run at the end of stage 1. Call it alone to rebuild the summary and group figures
**from disk**, without re-decoding.

In [ ]:
summary_df, payload = summarize_session(params_dict=PARAMS, paths=paths, label=LABEL)
summary_df

## Reading the output

The printed summary leads with the **generalisation gap**, then the group means. The row marked
`*` — `cross/*/c2u` — is the leak-free, deployable configuration and the only one here a live
system could reproduce.

Useful columns in `session_summary.csv`:

| column | meaning |
|---|---|
| `acc_mean_cross_<def>_c2u` | **the deployable cross-recording number** |
| `acc_mean_within_<def>_c2u` | same pipeline, recording-blind split (the control) |
| `delta_generalisation_gap_<def>_<mode>` | `within − cross` |
| `delta_leak_size_<split>_<def>` | `c2c − c2u`, i.e. what centering the test data buys |
| `train_sizes_<split>` / `test_sizes_<split>` | per-fold trial counts, so the match is auditable |
| `n_recordings` | folds for that subject |

Compare against `Full_Epoch_Analysis.ipynb`'s numbers, which use the same metric and window but
a trial-level split — the difference between them *is* the generalisation gap.